<a href="https://colab.research.google.com/github/Squad-Nina-da-Hora/wmc-desafio-previsao-demencia/blob/main/analise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
# **Análise de Risco de Alzheimer**
---


🎯 **Objetivo:**  Prever sinais de demência, através de informações clínicas e demográficas de pacientes com potencial risco de Alzheimer (OASIS).


---


Desafio Estatística com Python - Classificação

Squad Nina da Hora | Bootcamp Data Analytics 2026.1

## 1. Configurações Iniciais

Variáveis da base de dados:

- `Age`: Idade do paciente (numérico) 
- `Sex`: Gênero (F: feminino, M: masculino) 
- `EDUC`: Anos de escolaridade (numérico) 
- `SES`: Status socioeconômico (1 a 5) 
- `MMSE`: Escore do Mini Exame do Estado Mental (0 a 30) 
- `CDR`: Clinical Dementia Rating (0 a 3) 
- `eTIV`: Volume intracraniano estimado 
- `nWBV`: Proporção de volume cerebral normalizado 
- `ASF`: Fator de escala anatômica 
- `Group` (alvo): Classificação do paciente
  - `Nondemented` - será tratada para variável binária 0
  - `Demented` e `Converted` - serão tratadas para variável binária 1

In [ ]:
# ==============================
# IMPORTACOES
# ==============================

import kagglehub    # Para baixar datasets do Kaggle
import numpy as np  # Para operacoes numericas e arrays
import pandas as pd  # Para manipulaco e analise de data frames
from IPython.display import display, Markdown

# Bibliotecas para criacao de graficos
import seaborn as sns
import matplotlib.pyplot as plt
import missingno as msno

# Bibliotecas para criacao de modelos de ML
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedGroupKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import resample

# Carregamento da base de dados
arquivo = 'oasis_longitudinal'
url = f'{kagglehub.dataset_download("jboysen/mri-and-alzheimers")}/{arquivo}.csv'
df = pd.read_csv(url).drop(columns=['Subject ID', 'MRI ID', 'Visit', 'MR Delay', 'Hand'])

In [ ]:
# ==============================
# TRATAMENTO INICIAL
# ==============================

# 1. Renomeia algumas colunas
df.rename(columns={
  'Group': 'Demented',
  'M/F': 'Sex'
}, inplace=True)
# 2. Converte a coluna 'Demented' para bool
df['Demented'] = df['Demented'] != 'Nondemented'
# 3. Converte as colunas especificas para categorias
vars_categoricas = ['Sex', 'SES']
for cat in vars_categoricas:
  df[cat] = df[cat].astype('category')

df.head()

In [ ]:
# ==============================
# VARIAVEIS PARA REUTILIZACAO
# ==============================

alvo = 'Demented'

df_numericas_cols = df.select_dtypes(include=['number']).columns
df_categoricas_cols = df.select_dtypes(include=['str', 'object', 'category', 'bool']).columns

rotulos_vars = {
  'Demented': 'Status de Demência',
  'Sex': 'Gênero',
  'SES': 'Nível socioeconômico',
  'Age': 'Idade',
  'EDUC': 'Educação',
  'MMSE': 'Pontuação do Mini Exame Mental',
  'CDR': 'Classificação clínica de Demência',
  'eTIV': 'Volume intracraniano estimado',
  'nWBV': 'Volume cerebral normalizado',
  'ASF': 'Fator de escala anatômica'
}

alvo_rotulos = ['Sem Demência', 'Com Demência']

# Configuracos visuais dos graficos
paleta = 'flare'
cores = sns.color_palette(paleta, n_colors=2)

In [ ]:
# ==============================
# FUNCOES PARA REUTILIZACAO
# ==============================

def aplicar_cores(coluna, paleta=paleta):
  """
  Cria uma paleta de cores baseada na quantidade de itens unicos em uma coluna.

  - coluna: A coluna do DataFrame (ex: data['Group'])
  - paleta: O nome do estilo de cores (ex: 'flare')
  """
  return sns.color_palette(paleta, n_colors=coluna.nunique())

## 2. Análise Exploratória


In [ ]:
df.describe()

In [ ]:
df.describe(include=['object', 'category', 'bool'])

In [ ]:
contagem_nulos = df.isnull().sum()

datadict = pd.DataFrame(df.dtypes)
datadict.columns = ['tipos']
datadict['nulos'] = contagem_nulos
datadict['%_nulos'] = ((contagem_nulos / df.shape[0]) * 100).round(2)
datadict['unicos'] = df.nunique()
datadict

In [ ]:
# Grafico de nulos
msno.matrix(df, figsize=(10, 5), fontsize=9, color=sns.color_palette(paleta)[0])
plt.title('Visualização de Valores Ausentes', fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# Grafico de distribuicao das variaveis categoricas e alvo

num_plots = len(df_categoricas_cols)

fig, axes = plt.subplots(nrows=1, ncols=num_plots, figsize=(6 * num_plots, 6))

# Garante que 'axes' seja um array mesmo que haja apenas um subplot
if num_plots == 1:
  axes = [axes]

for i, col in enumerate(df_categoricas_cols):
  ax = sns.countplot(x=df[col], stat='percent', ax=axes[i], palette=aplicar_cores(df[col]), hue=df[col], legend=False)

  axes[i].set_xlabel('')
  axes[i].set_ylabel('Proporção de Pacientes (%)' if i == 0 else '') # Apenas o primeiro gráfico tera o rotulo Y

  axes[i].set_title(f'Distribuição de {rotulos_vars.get(col, col)}', fontsize=14, fontweight='bold')

  # Adiciona as porcentagens em cima das barras
  for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%')

plt.tight_layout() # Ajusta o layout para evitar sobreposicao
plt.show()

In [ ]:
# Boxplots das variaveis numericas vs. alvo

num_plots = len(df_numericas_cols)
ncols_3 = 3 # Define o numero de colunas desejado
nrows = (num_plots + ncols_3 - 1) // ncols_3 # Calcula o numero de linhas necessarias

fig, axes = plt.subplots(nrows=nrows, ncols=ncols_3, figsize=(6 * ncols_3, 6 * nrows))

# Achata o array 'axes' para facilitar a iteracao
axes = axes.flatten()

for i, col in enumerate(df_numericas_cols):
  ax = sns.boxplot(data=df, x=alvo, y=col, ax=axes[i], palette=paleta, hue=alvo, legend=False)

  axes[i].set_xticks([0, 1])
  axes[i].set_xticklabels(alvo_rotulos)
  axes[i].set_xlabel(f'{rotulos_vars.get(alvo, alvo)}')
  axes[i].set_title(f'Boxplot de {rotulos_vars.get(col, col)}', fontsize=14, fontweight='bold')

# Remove subplots vazios, se houver
for j in range(num_plots, nrows * ncols_3):
  fig.delaxes(axes[j])

plt.tight_layout() # Ajusta o layout para evitar sobreposicao
plt.show()

### Analise inicial

- Inicialmente, tratamos as categorias que deveriam ser booleanas ou categóricas, mas teremos que converter categóricas para numéricas.
- Verificamos que há duas variáveis com valores nulos, e são poucos, então serão tratados após a separação dos dados em treino e teste.
- Verificamos que os valores da variavel alvo são quase iguais, então pode ser que não precisemos ajustá-los com under/over sampling para os modelos.
- A amostra possui mais mulheres que homens e o maior nível socioeconômico é o 2. Não temos acesso ao significado os níveis, mas supomos que quanto maior o número, melhor é o nível do paciente.

## 3. Modelo de ML